<a href="https://colab.research.google.com/github/Aleeza-GH/ML-learning-journey/blob/main/Boost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import numpy as np
import pandas as pd

RANDOM_STATE = 42
N_ROWS = 1_000_000

rng = np.random.default_rng(RANDOM_STATE)

start_date = pd.Timestamp("2024-01-01")
end_date = pd.Timestamp("2025-01-01")

pickup_datetime = start_date + pd.to_timedelta(
    rng.integers(
        0,
        int((end_date - start_date).total_seconds()),
        N_ROWS
    ),
    unit="s"
)

trip_duration = np.clip(
    rng.lognormal(
        mean=np.log(18),
        sigma=0.55,
        size=N_ROWS
    ),
    3,
    120
)

dropoff_datetime = (
    pickup_datetime +
    pd.to_timedelta(trip_duration, unit="m")
)

trip_distance = np.clip(
    trip_duration * rng.normal(0.42, 0.10, N_ROWS)
    + rng.normal(0, 0.8, N_ROWS),
    0.5,
    45
)

passenger_count = rng.choice(
    [1, 2, 3, 4, 5, 6],
    size=N_ROWS,
    p=[0.58, 0.22, 0.10, 0.06, 0.03, 0.01]
)

pickup_zone = rng.choice(
    [
        "Manhattan",
        "Brooklyn",
        "Queens",
        "Bronx",
        "Staten_Island"
    ],
    size=N_ROWS,
    p=[0.48, 0.22, 0.20, 0.08, 0.02]
)

dropoff_zone = rng.choice(
    [
        "Manhattan",
        "Brooklyn",
        "Queens",
        "Bronx",
        "Staten_Island"
    ],
    size=N_ROWS,
    p=[0.46, 0.23, 0.21, 0.08, 0.02]
)

payment_type = rng.choice(
    [
        "Credit_Card",
        "Cash",
        "Digital_Wallet",
        "Other"
    ],
    size=N_ROWS,
    p=[0.68, 0.20, 0.09, 0.03]
)

traffic_level = rng.choice(
    [
        "Low",
        "Medium",
        "High"
    ],
    size=N_ROWS,
    p=[0.35, 0.45, 0.20]
)

hour = pickup_datetime.hour
day_of_week = pickup_datetime.dayofweek

is_weekend = (day_of_week >= 5).astype(int)

is_peak_hour = np.isin(
    hour,
    [7, 8, 9, 16, 17, 18, 19]
).astype(int)

zone_factor = np.select(
    [
        pickup_zone == "Manhattan",
        pickup_zone == "Brooklyn",
        pickup_zone == "Queens",
        pickup_zone == "Bronx",
        pickup_zone == "Staten_Island"
    ],
    [
        3.0,
        1.5,
        1.2,
        0.8,
        0.5
    ],
    default=1.0
)

traffic_factor = np.select(
    [
        traffic_level == "Low",
        traffic_level == "Medium",
        traffic_level == "High"
    ],
    [
        0.0,
        2.5,
        6.0
    ],
    default=0.0
)

payment_fee = np.where(
    payment_type == "Credit_Card",
    0.8,
    0.0
)

fare_amount = (
    3.5
    + trip_distance * 2.25
    + trip_duration * 0.32
    + passenger_count * 0.35
    + zone_factor
    + traffic_factor
    + is_peak_hour * 2.0
    + is_weekend * 0.75
    + payment_fee
    + rng.normal(0, 2.2, N_ROWS)
)

fare_amount = np.clip(
    fare_amount,
    5,
    250
)

df = pd.DataFrame({
    "pickup_datetime": pickup_datetime,
    "dropoff_datetime": dropoff_datetime,
    "passenger_count": passenger_count,
    "trip_distance": np.round(trip_distance, 2),
    "pickup_zone": pickup_zone,
    "dropoff_zone": dropoff_zone,
    "payment_type": payment_type,
    "traffic_level": traffic_level,
    "fare_amount": np.round(fare_amount, 2)
})

df.to_csv(
    "taxi_fare_regression_1M.csv",
    index=False
)

print(f"Dataset shape: {df.shape}")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print(df.head())

Dataset shape: (1000000, 9)
Rows: 1,000,000
Columns: 9
      pickup_datetime              dropoff_datetime  passenger_count  \
0 2024-02-02 15:58:49 2024-02-02 16:21:41.443595806                3   
1 2024-10-10 06:25:47 2024-10-10 06:53:13.909950080                2   
2 2024-08-27 13:45:22 2024-08-27 13:53:21.871376446                1   
3 2024-06-09 15:06:29 2024-06-09 15:17:33.739724444                1   
4 2024-06-07 11:36:20 2024-06-07 12:00:03.442912766                1   

   trip_distance pickup_zone dropoff_zone payment_type traffic_level  \
0          10.32      Queens    Manhattan  Credit_Card        Medium   
1          12.47   Manhattan    Manhattan  Credit_Card        Medium   
2           4.17   Manhattan     Brooklyn  Credit_Card           Low   
3           6.24   Manhattan    Manhattan  Credit_Card        Medium   
4           7.97       Bronx    Manhattan  Credit_Card           Low   

   fare_amount  
0        38.20  
1        45.34  
2        20.42  
3        27

In [16]:
import pandas as pd

DATA_PATH = "taxi_fare_regression_1M.csv"
TARGET = "fare_amount"

df = pd.read_csv(DATA_PATH)

print("Dataset Shape:", df.shape)
print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())

print("\nTarget Statistics:")
print(df[TARGET].describe())

print("\nSample Records:")
print(df.head())

Dataset Shape: (1000000, 9)

Data Types:
pickup_datetime      object
dropoff_datetime     object
passenger_count       int64
trip_distance       float64
pickup_zone          object
dropoff_zone         object
payment_type         object
traffic_level        object
fare_amount         float64
dtype: object

Missing Values:
pickup_datetime     0
dropoff_datetime    0
passenger_count     0
trip_distance       0
pickup_zone         0
dropoff_zone        0
payment_type        0
traffic_level       0
fare_amount         0
dtype: int64

Duplicate Rows: 0

Target Statistics:
count    1000000.000000
mean          36.350727
std           16.913632
min            5.000000
25%           24.890000
50%           32.620000
75%           43.540000
max          159.960000
Name: fare_amount, dtype: float64

Sample Records:
       pickup_datetime               dropoff_datetime  passenger_count  \
0  2024-02-02 15:58:49  2024-02-02 16:21:41.443595806                3   
1  2024-10-10 06:25:47  2024-10-10 

In [17]:
df["pickup_datetime"] = pd.to_datetime(df["pickup_datetime"])
df["dropoff_datetime"] = pd.to_datetime(df["dropoff_datetime"])

df["pickup_hour"] = df["pickup_datetime"].dt.hour
df["pickup_day"] = df["pickup_datetime"].dt.day
df["pickup_month"] = df["pickup_datetime"].dt.month
df["pickup_day_of_week"] = df["pickup_datetime"].dt.dayofweek

df["is_weekend"] = (
    df["pickup_day_of_week"] >= 5
).astype(int)

df["is_peak_hour"] = df["pickup_hour"].isin(
    [7, 8, 9, 16, 17, 18, 19]
).astype(int)

df["trip_duration_minutes"] = (
    df["dropoff_datetime"] -
    df["pickup_datetime"]
).dt.total_seconds() / 60

df["sort_datetime"] = df["pickup_datetime"]

In [18]:
TEST_SIZE = 0.20

df = df.sort_values(
    "sort_datetime"
).reset_index(drop=True)

split_index = int(
    len(df) * (1 - TEST_SIZE)
)

train_df = df.iloc[:split_index].copy()
test_df = df.iloc[split_index:].copy()

X_train = train_df.drop(
    columns=[
        TARGET,
        "sort_datetime",
        "pickup_datetime",
        "dropoff_datetime"
    ]
)

y_train = train_df[TARGET].copy()

X_test = test_df.drop(
    columns=[
        TARGET,
        "sort_datetime",
        "pickup_datetime",
        "dropoff_datetime"
    ]
)

y_test = test_df[TARGET].copy()

categorical_features = [
    "pickup_zone",
    "dropoff_zone",
    "payment_type",
    "traffic_level"
]

for column in categorical_features:
    X_train[column] = X_train[column].astype("category")
    X_test[column] = X_test[column].astype("category")

print("Training Features:", X_train.shape)
print("Testing Features:", X_test.shape)
print("Training Target:", y_train.shape)
print("Testing Target:", y_test.shape)

Training Features: (800000, 13)
Testing Features: (200000, 13)
Training Target: (800000,)
Testing Target: (200000,)


In [19]:
!pip install -q catboost lightgbm xgboost

In [20]:
from catboost import CatBoostRegressor

catboost_model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=42,
    verbose=100
)

catboost_model.fit(
    X_train,
    y_train,
    cat_features=categorical_features,
    eval_set=(X_test, y_test),
    use_best_model=True
)

0:	learn: 16.1466275	test: 16.1234547	best: 16.1234547 (0)	total: 1.33s	remaining: 22m 6s
100:	learn: 2.3206936	test: 2.3196602	best: 2.3196602 (100)	total: 1m 50s	remaining: 16m 27s
200:	learn: 2.2865650	test: 2.2901166	best: 2.2901166 (200)	total: 3m 30s	remaining: 13m 55s
300:	learn: 2.2806735	test: 2.2883327	best: 2.2883327 (300)	total: 5m 3s	remaining: 11m 44s
400:	learn: 2.2761387	test: 2.2868565	best: 2.2867407 (391)	total: 6m 49s	remaining: 10m 11s
500:	learn: 2.2730892	test: 2.2865967	best: 2.2865528 (435)	total: 8m 29s	remaining: 8m 27s
600:	learn: 2.2703598	test: 2.2866141	best: 2.2865134 (576)	total: 10m 6s	remaining: 6m 42s
700:	learn: 2.2676238	test: 2.2867130	best: 2.2864978 (631)	total: 11m 47s	remaining: 5m 1s
800:	learn: 2.2653727	test: 2.2868150	best: 2.2864978 (631)	total: 13m 20s	remaining: 3m 18s
900:	learn: 2.2629775	test: 2.2869959	best: 2.2864978 (631)	total: 14m 58s	remaining: 1m 38s
999:	learn: 2.2604823	test: 2.2876012	best: 2.2864978 (631)	total: 16m 34s	re

CatBoostRegressor(depth=8, eval_metric='RMSE', iterations=1000, learning_rate=0.05, loss_function='RMSE', random_seed=42, verbose=100)

In [22]:
catboost_predictions = catboost_model.predict(
    X_test
)

print("First 10 Predictions:")
print(catboost_predictions[:10])

First 10 Predictions:
[38.78550213 34.06053316 31.95579307 36.28774369 86.04794797 99.30184774
 11.5966832  42.48463222 34.09019114 19.61352519]


In [23]:
from lightgbm import LGBMRegressor

lightgbm_model = LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=64,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="regression",
    random_state=42,
    n_jobs=-1
)

lightgbm_model.fit(
    X_train,
    y_train,
    categorical_feature=categorical_features
)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.103878 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 616
[LightGBM] [Info] Number of data points in the train set: 800000, number of used features: 13
[LightGBM] [Info] Start training from score 36.357663


LGBMRegressor(colsample_bytree=0.8, learning_rate=0.05, n_estimators=1000,
              n_jobs=-1, num_leaves=64, objective='regression', random_state=42,
              subsample=0.8)

In [24]:
lightgbm_predictions = lightgbm_model.predict(
    X_test
)

print("First 10 Predictions:")
print(lightgbm_predictions[:10])

First 10 Predictions:
[ 39.08160111  34.03194754  31.91094057  36.12077286  85.72648918
 104.98527048  11.1201999   42.29694162  34.18593287  19.6134143 ]


In [25]:
import xgboost as xgb

X_train_xgb = pd.get_dummies(
    X_train,
    columns=categorical_features,
    dtype=float
)

X_test_xgb = pd.get_dummies(
    X_test,
    columns=categorical_features,
    dtype=float
)

X_test_xgb = X_test_xgb.reindex(
    columns=X_train_xgb.columns,
    fill_value=0
)

xgb_model = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_train_xgb,
    y_train
)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=8,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=1000,
             n_jobs=-1, num_parallel_tree=None, ...)

In [26]:
xgb_predictions = xgb_model.predict(
    X_test_xgb
)

print("First 10 Predictions:")
print(xgb_predictions[:10])

First 10 Predictions:
[ 39.129284  34.198204  31.901276  36.733128  86.15214  100.19255
  11.083398  42.347412  34.02743   19.670006]


In [27]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

results = []

models = {
    "CatBoost": catboost_predictions,
    "LightGBM": lightgbm_predictions,
    "XGBoost": xgb_predictions
}

for model_name, predictions in models.items():
    mae = mean_absolute_error(
        y_test,
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            predictions
        )
    )

    r2 = r2_score(
        y_test,
        predictions
    )

    results.append({
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "RMSE"
).reset_index(drop=True)

print(results_df)

      Model       MAE      RMSE        R2
0  LightGBM  1.789887  2.272837  0.981902
1  CatBoost  1.793774  2.286498  0.981684
2   XGBoost  1.805574  2.311909  0.981274


In [28]:
best_model_name = results_df.loc[
    results_df["RMSE"].idxmin(),
    "Model"
]

print("Best Model:", best_model_name)

best_predictions = lightgbm_predictions

comparison_df = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": best_predictions
})

comparison_df["Error"] = (
    comparison_df["Actual"] -
    comparison_df["Predicted"]
)

comparison_df["Absolute_Error"] = (
    comparison_df["Error"].abs()
)

print(comparison_df.head(10))

Best Model: LightGBM
   Actual   Predicted     Error  Absolute_Error
0   39.46   39.081601  0.378399        0.378399
1   29.26   34.031948 -4.771948        4.771948
2   31.27   31.910941 -0.640941        0.640941
3   34.82   36.120773 -1.300773        1.300773
4   84.93   85.726489 -0.796489        0.796489
5  105.52  104.985270  0.534730        0.534730
6   12.93   11.120200  1.809800        1.809800
7   44.61   42.296942  2.313058        2.313058
8   35.30   34.185933  1.114067        1.114067
9   18.40   19.613414 -1.213414        1.213414


In [29]:
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": lightgbm_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    "Importance",
    ascending=False
).reset_index(drop=True)

print(feature_importance)

                  Feature  Importance
0           trip_distance       12159
1   trip_duration_minutes       10653
2              pickup_day        7574
3             pickup_hour        6635
4      pickup_day_of_week        5207
5            pickup_month        4974
6           traffic_level        4141
7         passenger_count        3843
8             pickup_zone        2501
9            is_peak_hour        1732
10           payment_type        1591
11           dropoff_zone        1578
12             is_weekend         412


In [30]:
MODEL_PATH = "lightgbm_fare_model.txt"

lightgbm_model.booster_.save_model(
    MODEL_PATH
)

print(f"Model saved successfully: {MODEL_PATH}")

Model saved successfully: lightgbm_fare_model.txt


In [31]:
import lightgbm as lgb

loaded_model = lgb.Booster(
    model_file=MODEL_PATH
)

print("Model loaded successfully.")

Model loaded successfully.


In [32]:
new_ride = pd.DataFrame({
    "passenger_count": [2],
    "trip_distance": [8.5],
    "pickup_zone": ["Manhattan"],
    "dropoff_zone": ["Brooklyn"],
    "payment_type": ["Credit_Card"],
    "traffic_level": ["High"],
    "pickup_hour": [18],
    "pickup_day": [14],
    "pickup_month": [8],
    "pickup_day_of_week": [4],
    "is_weekend": [0],
    "is_peak_hour": [1],
    "trip_duration_minutes": [25]
})

for column in categorical_features:
    new_ride[column] = new_ride[column].astype("category")

prediction = loaded_model.predict(
    new_ride
)

print(f"Predicted Fare: ${prediction[0]:.2f}")

Predicted Fare: $42.98
